# 01 — MCP / toolkit market context

Validates the existing read-only data layer the dashboard, the Gemini analyst and the
MCP server share (`qbs/agent/toolkit.py`), as a pre-deployment check on the market
context the lab exposes.

**Expected behaviour**
- every offline tool returns non-empty text the package computed;
- `market_snapshot` and `stock_data` agree on the date the data stands on, and both
  report how many sessions behind it is (freshness);
- a ticker outside the focus list gets an explicit refusal, not invented data;
- no tool can place, modify or cancel an order, and none writes trading data.

The trading agent itself gives its model **no tools at all** (checked in the last
cell); this notebook validates the shared data layer it sits next to.

**Modes.** `MOCK` (default) makes zero external LLM calls. `REAL_LLM` needs
`QBS_AGENT_TEST_MODE=REAL_LLM` **and** `QBS_AGENT_TEST_ALLOW_PAID=1`, prints a cost
estimate first, and stops at `QBS_AGENT_TEST_MAX_CALLS` (default 20).

**Isolation.** Every run uses a fresh temporary database (`tk.workspace()`); the
production config (`var/agent/config.json`) and log (`var/agent/agent.db`) are never
read or written, and nothing here can reach `qbs.live` or a broker.

**Dependencies.** `pip install -r requirements.txt jupyter` (MOCK). REAL_LLM also needs
the selected provider's package and key (see docs/TRADING_AGENT.md).

In [1]:
import os, sys, pathlib, json, logging
here = pathlib.Path.cwd()
ROOT = next(p for p in [here, *here.parents] if (p / "qbs" / "trading_agent").is_dir())
sys.path.insert(0, str(ROOT))
logging.basicConfig(level=logging.WARNING)

import pandas as pd
from qbs.trading_agent import testkit as tk

MODE = tk.test_mode()          # MOCK unless QBS_AGENT_TEST_MODE=REAL_LLM and QBS_AGENT_TEST_ALLOW_PAID=1
print("repo:", ROOT)
print("MODE:", MODE, "| paid-call cap:", tk.max_calls() if MODE == tk.REAL_LLM else 0)

repo: /home/user/qqq_boxx_strategies
MODE: MOCK | paid-call cap: 0


## Build the toolkit over a fixed, offline book

Synthetic prices with a fixed seed (`qbs.data.synthetic_prices`), so the run is reproducible and needs no network. Web tools are removed (`allow_web=False`), not merely blocked.

In [2]:
from qbs.agent import evidence as ev, toolkit as tkit
from qbs.config import Config
from qbs.data import synthetic_prices
from qbs.universe import synthetic_universe

px = synthetic_prices()
uni = synthetic_universe(n=40).reindex(px.index).ffill()
book = ev.Book(universe=uni, prices=px, cfg=Config(), note="synthetic fixture")
data = tkit.Data(book=book, watchlist=[], ohlc_loader=lambda t: None)
tools = {t.__name__: t for t in tkit.build_tools(data, allow_web=False)}
check = tk.Check("01 MCP market context")
print(sorted(tools))

['breakout_funnel', 'breakout_trades', 'current_picks', 'fundamentals', 'list_universe', 'market_overview', 'market_snapshot', 'name_momentum', 'price_action', 'reload_data', 'sector_leadership', 'stock_data', 'stock_vs_market', 'strategy_performance']


## Every offline tool returns text

In [3]:
ticker = book.universe.columns[0]
calls = {"current_picks": (), "stock_data": (ticker,), "name_momentum": (ticker,),
         "price_action": (ticker, 3), "list_universe": (), "market_overview": (3,),
         "market_snapshot": (), "sector_leadership": (), "stock_vs_market": (ticker,),
         "breakout_funnel": (), "breakout_trades": ()}
rows = []
for name, args in calls.items():
    out = tools[name](*args)
    ok = isinstance(out, str) and bool(out.strip()) and "failed" not in out.split("\n")[0]
    check(f"{name} returns text", ok, out.split("\n")[0][:80])
    rows.append({"tool": name, "chars": len(out), "first_line": out.split("\n")[0][:70]})
pd.DataFrame(rows)

,tool,chars,first_line
0,current_picks,945,CURRENT PICKS — Data as of 2026-09-08 (22 trad...
1,stock_data,2107,STOCK_CONTEXT
2,name_momentum,1971,MOMENTUM PROFILE — SY000
3,price_action,2242,PRICE ACTION — SY000
4,list_universe,378,"40 names, prices through 2026-09-08:"
5,market_overview,2567,MARKET OVERVIEW — NASDAQ-100 FALLBACK — an ind...
6,market_snapshot,2127,MARKET_CONTEXT
7,sector_leadership,182,No sector map in this run: the Nasdaq-100 cach...
8,stock_vs_market,648,SY000 IN THE MARKET — NASDAQ-100 FALLBACK — an...
9,breakout_funnel,107,No funnel was passed in. Build one with `qbs.b...


## Snapshot consistency and freshness

In [4]:
snap = tkit.split_json(tools["market_snapshot"]())
stock = tkit.split_json(tools["stock_data"](ticker))
check("market_snapshot parses as JSON", isinstance(snap, dict))
check("stock_data parses as JSON", isinstance(stock, dict))
check("same data_as_of", snap["data_as_of"] == stock["data_as_of"],
      f'{snap["data_as_of"]} vs {stock["data_as_of"]}')
check("freshness is reported", "sessions_behind" in snap and "sessions_behind" in stock,
      f'sessions_behind={snap.get("sessions_behind")}')
check("caveats travel with the numbers", bool(snap.get("interpretation_limits")))
print(json.dumps({k: snap[k] for k in ("analysis_date", "data_as_of", "sessions_behind",
                                       "universe", "sample_size")}, indent=1))
if snap["sessions_behind"]:
    print(f"NOTE: the fixture is {snap['sessions_behind']} sessions old -- the tools say so "
          "instead of presenting it as current, which is the behaviour under test.")

{
 "analysis_date": "2026-10-09",
 "data_as_of": "2026-09-08",
 "sessions_behind": 22,
 "universe": "Nasdaq-100 constituents -- FALLBACK, an index and not the market",
 "sample_size": 40
}
NOTE: the fixture is 22 sessions old -- the tools say so instead of presenting it as current, which is the behaviour under test.


## Missing data is refused, not invented

In [5]:
from qbs.agent.context import NOT_IN_FOCUS_MESSAGE
out = tools["stock_data"]("ZZZZ")
check("unknown ticker refused", out.startswith("NOT IN FOCUS LIST") and NOT_IN_FOCUS_MESSAGE in out)
check("refusal is not parsed as data", tkit.split_json(out) == out)
out2 = tools["price_action"]("ZZZZ", 3)
print(out[:120]); print(out2[:160])

NOT IN FOCUS LIST: ZZZZ. Reply to the user with exactly: The stock is not in our focused list, please add it to watchlis
No prices for ZZZZ: it is not in the cached universe (40 names) and not on the watchlist. 


## The tools are read-only

Three independent checks: no tool name is an order verb; the toolkit and the MCP server import nothing from `qbs.live` or a broker client; and calling every tool leaves the trading data on disk untouched.

In [6]:
import ast, re, hashlib

verbs = re.compile(r"order|submit|cancel|place|execute|modify|buy|sell", re.I)
check("no order-verb tool names", not [n for n in tools if verbs.search(n)],
      ", ".join(n for n in tools if verbs.search(n)))

def imports(path):
    tree = ast.parse(open(path, encoding="utf-8").read())
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            yield from (a.name for a in node.names)
        elif isinstance(node, ast.ImportFrom):
            yield "." * node.level + (node.module or "")

for f in ("qbs/agent/toolkit.py", "qbs/agent/mcp_server.py"):
    bad = [n for n in imports(ROOT / f)
           if "live" in n.lstrip(".").split(".") or n.split(".")[0] in ("ib_async", "ib_insync")]
    check(f"{f} imports no broker code", not bad, ", ".join(bad))

def digest(paths):
    return {str(p): hashlib.sha256(p.read_bytes()).hexdigest()
            for p in paths if p.is_file()}
watched = list((ROOT / "data").glob("*.csv")) + list((ROOT / "var").glob("*.db"))
before = digest(watched)
for name, args in calls.items():
    tools[name](*args)
check("trading data untouched by the tools", digest(watched) == before,
      f"{len(before)} files hashed")

True

## The trading agent's model gets no tools

In [7]:
from qbs.trading_agent import session as ss
from qbs.trading_agent.engine import TradingAgentEngine
from qbs.trading_agent.market_context import MarketContextBuilder

fx = tk.load_fixture()
ws = tk.workspace(tk.MOCK)            # always MOCK here: this check needs no model
clock = tk.Clock(tk.et(fx.replay_date, 9, 0))
ss.save_daily_prompt(ws.cfg, fx.prompt, clock()); ss.authorize(ws.cfg, clock())
clock.set(tk.et(fx.replay_date, 10, 16))
sess, _ = ss.activate(ws.cfg, clock()); sess.transition(ss.RUNNING, "nb01")
llm = tk.make_provider(tk.MOCK, ws.cfg)
eng = TradingAgentEngine(ws.cfg, sess, llm, MarketContextBuilder(tk.FixtureBars(fx, clock)))
eng.run_cycle(tk.et(fx.replay_date, 10, 15))
check("LLM calls carry only system + user + settings",
      all(set(c) == {"system", "user", "settings"} for c in llm.calls), f"{len(llm.calls)} calls")

True

## Result

In [8]:
res = check.frame()
display(res)
print(tk.save_results("01_mcp_market_context", res))
check.raise_if_failed()

,check,result,detail
0,current_picks returns text,PASS,CURRENT PICKS — Data as of 2026-09-08 (22 trad...
1,stock_data returns text,PASS,STOCK_CONTEXT
2,name_momentum returns text,PASS,MOMENTUM PROFILE — SY000
3,price_action returns text,PASS,PRICE ACTION — SY000
4,list_universe returns text,PASS,"40 names, prices through 2026-09-08:"
5,market_overview returns text,PASS,MARKET OVERVIEW — NASDAQ-100 FALLBACK — an ind...
6,market_snapshot returns text,PASS,MARKET_CONTEXT
7,sector_leadership returns text,PASS,No sector map in this run: the Nasdaq-100 cach...
8,stock_vs_market returns text,PASS,SY000 IN THE MARKET — NASDAQ-100 FALLBACK — an...
9,breakout_funnel returns text,PASS,No funnel was passed in. Build one with `qbs.b...


/home/user/qqq_boxx_strategies/notebooks/agent_tests/results/01_mcp_market_context_20261009T093311Z.csv


**Troubleshooting**
- *A tool's first line says "failed"*: the data layer raised; the text names the cause.
  With real data, build the cache first (`python run_backtest.py`).
- *Trading data changed*: something wrote into `data/` or `var/` during the run — a
  regression in the read-only contract; find the tool with a bisection over `calls`.
- *Fixture hash error*: `notebooks/agent_tests/fixtures/v1` was edited; rebuild a NEW
  version with `scripts/build_agent_fixtures.py` instead.